# Space-Grade Fault Detector — harden `top` as one macro (Classic flow)

Adapted from the chipathon-2026 `01_rtl2gds_counter.ipynb` skeleton. Differences from that notebook:
- RTL already exists in your repo (`rtl/*.v`, 11 modules) — not inlined here.
- Runs **directly** in-container via `subprocess` — no `docker exec` hop, since the kernel is already inside `gf180`.
- Uses a real `constraints.sdc` (`FALLBACK_SDC`) instead of relying on LibreLane's default `base.sdc`.
- No padring, no chip-top wrapper yet — that's the next notebook. This one only produces a hardened `top` macro: `gds/lef/lib/v`.

RUN_LIBRELANE defaults to False — flip it once Step 1's port names are confirmed.

## Step 0 — configuration

In [ ]:
from pathlib import Path
import subprocess, textwrap, csv, yaml

RUN_LIBRELANE = True   

# One filesystem, one kernel — you're already inside gf180. No host/container split.
PROJECT_DIR = Path("/foss/designs/Space-Grade-Mechanical-Fault-Detector")
PDK_ROOT     = "/foss/pdks"
PDK_NAME     = "gf180mcuD"
STD_CELL_LIB = "gf180mcu_fd_sc_mcu7t5v0"

DESIGN_NAME  = "top"
RUN_TAG      = "macro_v1"
CLOCK_PERIOD = 50 #62.5ns -> 16 MHz

print("Project dir:", PROJECT_DIR)
print("Exists:", PROJECT_DIR.exists())

## Step 1 — confirm CLOCK_PORT / RESET_PORT from top.v
Don't guess these into the config below. Read the actual port declaration first.

In [ ]:
top_v = (PROJECT_DIR / "rtl" / "top.v").read_text()
print(top_v[:2000])

# Fill these in from the printed module port list before running Step 3:
CLOCK_PORT = "clk"
RESET_PORT = "sys_rst_n"   # used in Step 2 to patch constraints.sdc

## Step 2 — confirm constraints.sdc, patch in the reset port
Upload `constraints.sdc` directly into `PROJECT_DIR` via the Jupyter file browser (drag-and-drop into the
`Space-Grade-Mechanical-Fault-Detector` folder on the left panel) — no copy-from-elsewhere step needed
since everything is already one filesystem. This cell just patches the reset line in place.

In [ ]:
sdc_path = PROJECT_DIR / "constraints.sdc"

if not sdc_path.exists():
    print(f"Not found: {sdc_path} — upload it via the Jupyter file browser first, then re-run this cell.")
else:
    sdc_text = sdc_path.read_text()
    if "REPLACE" not in RESET_PORT:
        sdc_text = sdc_text.replace(
            "# set_false_path -from [get_ports RESET_PORT_NAME]",
            f"set_false_path -from [get_ports {RESET_PORT}]"
        )
        sdc_path.write_text(sdc_text)
        print(f"Patched reset false-path into {sdc_path}")
    else:
        print("RESET_PORT still a placeholder — fill it in from Step 1 before patching.")

## Step 3 — pre-flight: confirm TMR keep-attributes are in the RTL
`(* keep *)` / `dont_touch` must already be present in `tmr_reg_bank.v`, `axis_sequencer.v`,
`goertzel_core.v`, `magnitude_compute.v` — otherwise Yosys will merge the redundant register
copies during Step 5 and TMR will silently disappear from the netlist.

In [ ]:
for fname in ["tmr_reg_bank.v", "axis_sequencer.v", "goertzel_core.v", "magnitude_compute.v"]:
    text = (PROJECT_DIR / "rtl" / fname).read_text()
    has_keep = "keep" in text or "dont_touch" in text
    print(f"{fname:25s} keep/dont_touch present: {has_keep}")

In [ ]:
# explore_script = f"""
# set -euo pipefail
# cd {PROJECT_DIR}
# source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
# librelane --flow SynthesisExploration config.yaml \\
#     --pdk {PDK_NAME} \\
#     --pdk-root {PDK_ROOT} \\
#     --manual-pdk \\
#     --run-tag explore_v1
# """
# proc = subprocess.run(["bash", "-lc", explore_script], capture_output=True, text=True, timeout=None)
# print(proc.stdout[-6000:])

## Step 4 — run ENTIRE LibreLane (direct subprocess, no docker exec)

In [ ]:
RUN_TAG = "macro_v6"

flow_script = textwrap.dedent(f"""
    set -euo pipefail
    cd {PROJECT_DIR}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    librelane config.yaml \\
        --pdk {PDK_NAME} \\
        --pdk-root {PDK_ROOT} \\
        --manual-pdk \\
        --run-tag {RUN_TAG}
""").strip()

print("$", flow_script)
if RUN_LIBRELANE:
    proc = subprocess.run(["bash", "-lc", flow_script], capture_output=True, text=True, timeout=None)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR:", proc.stderr[-2000:])
    print("returncode:", proc.returncode)
else:
    print("(skipped — flip RUN_LIBRELANE)")

## Step 5 — read metrics
Key is `design__die__area` (no `__um2` suffix) — confirmed from the validated counter run.

In [ ]:
metrics_path = PROJECT_DIR / "runs" / RUN_TAG / "final" / "metrics.csv"
wanted = [
    "design__die__area",
    "design__instance__count__stdcell",
    "timing__setup_vio__count",
    "timing__hold_vio__count",
    "magic__drc_error__count",
    "klayout__drc_error__count",
    "design__lvs_error__count",
    "power__total",
]

found = {}
if metrics_path.exists():
    with metrics_path.open() as fh:
        for row in csv.reader(fh):
            if row and row[0] in wanted:
                found[row[0]] = row[1] if len(row) > 1 else ""
    for key in wanted:
        print(f"  {key:35s} {found.get(key, '(missing)')}")
else:
    print(f"Not found: {metrics_path} — Step 5 hasn't completed yet.")

In [ ]:
run_dir = PROJECT_DIR / "runs" / RUN_TAG

for rpt in sorted(run_dir.rglob("*.rpt")):
    if "reports" not in rpt.parts:
        continue
    n = rpt.read_text(errors="ignore").count("sys_rst_n")
    if n:
        print(f"{n:4d}  {rpt.relative_to(run_dir)}")

In [ ]:
def list_reports(run_dir, keyword=None):
    run_dir = Path(run_dir)
    for step_dir in sorted(run_dir.iterdir()):
        reports_dir = step_dir / "reports"
        if not step_dir.is_dir() or not reports_dir.exists():
            continue
        files = [f for f in sorted(reports_dir.rglob("*")) if f.is_file()]
        if keyword:
            files = [f for f in files if keyword.lower() in f.name.lower()]
        if files:
            print(f"### {step_dir.name}")
            for f in files:
                print("   ", f.relative_to(run_dir))

list_reports(run_dir)
list_reports(run_dir, keyword="wns")

In [ ]:
sta_steps = [d for d in sorted(run_dir.iterdir())
             if d.is_dir() and any(k in d.name.lower() for k in
                                    ["staprepnr", "stamidpnr", "stapostpnr", "resizertiming"])]

for step_dir in sta_steps:
    print(f"=== {step_dir.name} ===")
    for wns in sorted(step_dir.rglob("*wns*.rpt")):
        print("  ", wns.relative_to(run_dir), "->", wns.read_text(errors="ignore").strip().splitlines()[-1])

In [ ]:
#checking worst case path

worst_path = run_dir / "55-openroad-stapostpnr" / "nom_ss_125C_4v50" / "max.rpt"
print(worst_path.read_text()[-3000:])

In [ ]:
import re
text = worst_path.read_text()
weak_drivers = re.findall(r"(\S+_1)\s*\(gf180mcu_fd_sc_mcu7t5v0__\w+\)", text)
print(f"{len(weak_drivers)} minimum-size-cell instances in this single path:")
print(weak_drivers)

## Step 6 — confirm TMR flop count survived synthesis
Check the Yosys stats report before treating this run as done.

In [ ]:
run_dir = PROJECT_DIR / "runs" / RUN_TAG
stat_rpts = sorted(run_dir.glob("*-yosys-synthesis/reports/*.stat.rpt")) if run_dir.exists() else []
for p in stat_rpts:
    print(p)
print("Open the report above and confirm tmr_reg_bank flop count is 3x the single-copy count, not deduplicated.")

## Next
Once Step 5/6 are clean (all signoff metrics zero, TMR intact): this `top` macro's
`gds/lef/lib/v` quartet feeds the next notebook — chip-top + padring, following the
`02`/`03` pattern (`MACROS:` dict, `PDN_MACRO_CONNECTIONS:`, `SLOT=` selection).

In [55]:
klayout_call = f"""
    klayout -b -r runs/{RUN_TAG}/final/top.gds -rd run_tag={RUN_TAG} -rd pdk_name={PDK_NAME} -rd std_cell_lib={STD_CELL_LIB}
    """
proc = subprocess.run(["bash", "-lc", klayout_call], capture_output=True, text=True, timeout=None,executable='/bin/bash')
print(proc.stdout[-6000:])